# Community Detection

### Import libraries

In [2]:
import math
import numpy as np
import pandas as pd
import networkx as nx

import matplotlib.pyplot as plt
import seaborn as sns
import time
import dimod
from networkx.algorithms.community.quality import modularity as nx_modularity
from dwave.system import LeapHybridSampler
from dwave.system import DWaveSampler, EmbeddingComposite
from dwave.system import FixedEmbeddingComposite
from dwave.system import LeapHybridDQMSampler
from dwave.samplers import SimulatedAnnealingSampler
from dwave.embedding import embed_bqm
from dwave.embedding.chain_strength import uniform_torque_compensation
from minorminer import find_embedding
from dimod import ExactSolver

# To extract, save, and reuse embedding
import json
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

### Load data

In [3]:
# Load preprocessed network
G = nx.read_graphml('neuronal_graphs/neurons10.graphml')
G.remove_nodes_from(list(nx.isolates(G)))

print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Graph type: {type(G)}")
print(f"Is directed: {G.is_directed()}")

# Display sample node and edge attributes
sample_node = list(G.nodes())[0]
sample_edge = list(G.edges(data=True))[0]
print(f"\nSample node: {sample_node}")
print(f"Sample edge: {sample_edge}")

Graph loaded: 77 nodes, 96 edges
Graph type: <class 'networkx.classes.digraph.DiGraph'>
Is directed: True

Sample node: 3
Sample edge: ('3', '21', {'weight': -0.1004582051503421})


In [5]:
list(G.nodes())

['3',
 '5',
 '6',
 '7',
 '10',
 '12',
 '13',
 '14',
 '15',
 '17',
 '19',
 '21',
 '26',
 '27',
 '30',
 '32',
 '33',
 '34',
 '35',
 '36',
 '37',
 '41',
 '42',
 '43',
 '45',
 '47',
 '48',
 '50',
 '51',
 '52',
 '57',
 '58',
 '61',
 '66',
 '70',
 '72',
 '76',
 '84',
 '85',
 '87',
 '88',
 '90',
 '93',
 '94',
 '95',
 '97',
 '99',
 '102',
 '104',
 '106',
 '111',
 '113',
 '114',
 '115',
 '118',
 '120',
 '126',
 '128',
 '133',
 '139',
 '140',
 '142',
 '144',
 '148',
 '149',
 '150',
 '153',
 '154',
 '156',
 '158',
 '163',
 '164',
 '165',
 '166',
 '167',
 '168',
 '169']

# 1 Graph preprocessing

### 1.1 Conversion to adjacency matrix

In [3]:
# Convert directed graph to undirected
G_undirected = G.to_undirected()

# Remove any remaining self-loops
G_undirected.remove_edges_from(nx.selfloop_edges(G_undirected))

# Create unweighted adjacency matrix
A = nx.adjacency_matrix(G_undirected, weight=None).toarray()
N = G_undirected.number_of_nodes()
M = G_undirected.number_of_edges()  # Total number of edges

print(f"\nUndirected graph: {N} nodes, {M} edges")
print(f"Adjacency matrix shape: {A.shape}")
print(f"Sparsity: {(A == 0).sum() / A.size * 100:.2f}% zeros")
print(f"\nDegree statistics:")
degrees = [G_undirected.degree(n) for n in G_undirected.nodes()]
print(f"  Min: {min(degrees)}, Max: {max(degrees)}, Mean: {np.mean(degrees):.2f}")


Undirected graph: 77 nodes, 48 edges
Adjacency matrix shape: (77, 77)
Sparsity: 98.38% zeros

Degree statistics:
  Min: 1, Max: 3, Mean: 1.25


# 2 Classical community detection

In [4]:
def communities_to_partition(communities):
    """
    Convert a list of community sets to a partition array.
    Returns: array where partition[i] = community_id for node i
    """
    node_list = list(G_undirected.nodes())  # Use the current graph's node order
    node_to_idx = {node: idx for idx, node in enumerate(node_list)}
    partition = np.zeros(len(node_list), dtype=int)
    for community_id, community_set in enumerate(communities):
        for node in community_set:
            partition[node_to_idx[node]] = community_id
    return partition

def compute_modularity(partition, A):
    """
    Compute modularity Q for a given partition.
    Q = (1/2m) * sum_ij (A_ij - k_i*k_j/2m) * delta(c_i, c_j)
    where k_i is the degree of node i, and delta(c_i, c_j) = 1 if nodes i,j in same community
    """
    n = A.shape[0]

    degrees = A.sum(axis=1)
    m = A.sum() / 2.0  # total edges (double counted in adjacency matrix)
    
    Q = 0.0
    for i in range(n):
        for j in range(n):
            if partition[i] == partition[j]:
                Q += A[i, j] - (degrees[i] * degrees[j]) / (2.0 * m)
    
    Q /= (2.0 * m)
    return Q

def compute_conductance(partition, A):
    """
    Compute conductance for a given partition.
    Conductance = cut(S, S') / min(vol(S), vol(S'))
    where cut(S, S') is the number of edges between S and its complement,
    and vol(S) is the sum of degrees in S.
    """
    degrees = A.sum(axis=1)
    conductances = []
    for community_id in np.unique(partition):
        S = np.where(partition == community_id)[0]

        # Skip single-node communities (undefined conductance)
        if len(S) <= 1:
            continue

        S_complement = np.where(partition != community_id)[0]
        cut = A[np.ix_(S, S_complement)].sum()
        vol_S = degrees[S].sum()
        vol_S_complement = degrees[S_complement].sum()

        if min(vol_S, vol_S_complement) > 0:
            conductance = cut / min(vol_S, vol_S_complement)
            conductances.append(conductance)

    return np.mean(conductances) if conductances else 0.0

### 2.1 Louvain

In [5]:
start_time = time.time()
communities_louvain = list(nx.community.louvain_communities(G_undirected, seed=42))
louvain_runtime = time.time() - start_time

partition_louvain = communities_to_partition(communities_louvain)
modularity_louvain = compute_modularity(partition_louvain, A)
conductance_louvain = compute_conductance(partition_louvain, A)

print(f"Runtime: {louvain_runtime:.4f}s")
print(f"Number of communities: {len(communities_louvain)}")
print(f"Modularity: {modularity_louvain:.6f}")
print(f"Conductance: {conductance_louvain:.6f}")
print(f"Community sizes: {sorted([len(c) for c in communities_louvain], reverse=True)}")
print(f"Number of communities: {len(communities_louvain)}")

Runtime: 0.0281s
Number of communities: 52
Modularity: 0.488281
Conductance: 0.155556
Community sizes: [5, 4, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Number of communities: 52


### 2.2 Greedy modularity optimization

In [6]:
start_time = time.time()
communities_greedy = list(nx.community.greedy_modularity_communities(G_undirected))
greedy_runtime = time.time() - start_time

partition_greedy = communities_to_partition(communities_greedy)
modularity_greedy = compute_modularity(partition_greedy, A)
conductance_greedy = compute_conductance(partition_greedy, A)

print(f"Runtime: {greedy_runtime:.4f}s")
print(f"Number of communities: {len(communities_greedy)}")
print(f"Modularity: {modularity_greedy:.6f}")
print(f"Conductance: {conductance_greedy:.6f}")
print(f"Community sizes: {sorted([len(c) for c in communities_greedy], reverse=True)}")

Runtime: 0.0120s
Number of communities: 29
Modularity: 0.950521
Conductance: 0.000000
Community sizes: [6, 5, 5, 4, 4, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


# 3 QUBO with Modularity

### 3.0 Build QUBO matrix

In [7]:
def build_modularity_qubo(A, num_communities=None, lambda_penalty=None, verbose=False):
    """
    Build QUBO matrix for modularity maximization via binary community assignment.
    Variables: x_{i,c} for i in nodes, c in communities
    Constraint: Each node assigned to exactly one community (handled via penalty)
    
    One-hot encoding:
    - num_binary_vars = N * num_communities
    - x[i*num_communities + c] = 1 if node i in community c
    
    QUBO objective: minimize -Q + lambda * penalty
    
    Parameters:
    -----------
    A : ndarray
        Adjacency matrix
    num_communities : int, optional
        Number of communities (default: sqrt(N))
    lambda_penalty : float, optional
        Penalty weight for one-hot constraint.
        If None, automatically set to 10x the max modularity contribution.
    verbose : bool
        Print diagnostics
    """

    if num_communities is None:
        num_communities = max(2, int(np.sqrt(N)))

    n_vars = N * num_communities 
    Q = {}

    degrees = A.sum(axis=1)
    m = A.sum() / 2  # Total edges

    # Modularity term: Q = (1/2m) * sum_{c} sum_{i,j} B_{ij} x_{i,c} x_{j,c}
    # For minimization, use -Q in QUBO objective
    max_B = 0  # Track max modularity contribution for scaling
    
    for c in range(num_communities):
        for i in range(N):
            for j in range(i, N):
                var_i = i * num_communities + c
                var_j = j * num_communities + c
                B_ij = A[i, j] - (degrees[i] * degrees[j]) / (2 * m)
                coeff = -B_ij / (2 * m)
                max_B = max(max_B, abs(coeff))

                if i == j:  # diagonal
                    if var_i not in Q:
                        Q[var_i] = 0.0
                    Q[var_i] += coeff
                else:  # off-diagonal
                    key = (var_i, var_j) if var_i < var_j else (var_j, var_i)
                    if key not in Q:
                        Q[key] = 0.0
                    Q[key] += coeff

    if lambda_penalty is None:
        lambda_penalty = max(10.0 * max_B, 5.0)  # At least 5.0
    
    if verbose:
        print(f"Modularity scale (max |coeff|): {max_B:.6f}")
        print(f"Penalty weight (lambda): {lambda_penalty:.6f}")

    # Constraint penalty: each node in exactly one community
    # Penalty: lambda * (sum_c x_{i,c} - 1)^2
    # Expansion:
    #   (sum_c x_{i,c} - 1)^2 = sum_c x_{i,c}^2 + 2*sum_{c1<c2} x_{i,c1}*x_{i,c2} - 2*sum_c x_{i,c} + 1
    #   = sum_c x_{i,c} + 2*sum_{c1<c2} x_{i,c1}*x_{i,c2} - 2*sum_c x_{i,c} + 1   [since x^2=x for binary]
    #   = -sum_c x_{i,c} + 2*sum_{c1<c2} x_{i,c1}*x_{i,c2} + 1
    
    # Linear coefficient: -lambda per variable
    # Quadratic coefficient: +2*lambda per pair

    const_term = N * lambda_penalty
    
    for i in range(N):
        # Linear terms: -lambda * x_{i,c}
        for c in range(num_communities):
            var_i = i * num_communities + c
            if var_i not in Q:
                Q[var_i] = 0.0
            Q[var_i] += -lambda_penalty

        # Quadratic terms: +2*lambda * x_{i,c1}*x_{i,c2}
        for c1 in range(num_communities):
            for c2 in range(c1 + 1, num_communities):
                var_1 = i * num_communities + c1
                var_2 = i * num_communities + c2
                key = (var_1, var_2) if var_1 < var_2 else (var_2, var_1)
                if key not in Q:
                    Q[key] = 0.0
                Q[key] += 2.0 * lambda_penalty

    Q['__const__'] = const_term
    
    if verbose:
        print(f"Constant term (N*lambda): {const_term:.2f}")
        print(f"Total QUBO terms: {len(Q)-1} + 1 const")  # -1 for const term itself

    return Q, num_communities

In [8]:
# Estimate reasonable number of communities
num_communities_max = max(len(communities_louvain), len(communities_greedy))
print(f"Target number of communities: {num_communities_max}")
print(f"Total QUBO variables: {N * num_communities_max}")

# Build QUBO
print(f"\nBuilding QUBO matrix...")
Q_dict, k_communities = build_modularity_qubo(A, num_communities_max, verbose=True)
print(f"QUBO built with {len(Q_dict)-1} terms")
print(f"Constant term: {Q_dict.get('__const__', 0):.2f}")

Target number of communities: 52
Total QUBO variables: 4004

Building QUBO matrix...
Modularity scale (max |coeff|): 0.010308
Penalty weight (lambda): 5.000000
Constant term (N*lambda): 385.00
Total QUBO terms: 258258 + 1 const
QUBO built with 258258 terms
Constant term: 385.00


In [9]:
# Convert QUBO to BQM (Binary Quadratic Model) for D-Wave
Q_dict_clean = {}
const_offset = 0

for k, v in Q_dict.items():
    if k == '__const__':
        # Extract constant term for later addition
        const_offset = float(v)
    elif isinstance(k, tuple):
        Q_dict_clean[k] = float(v)
    else:
        node_idx = int(k)
        Q_dict_clean[(node_idx, node_idx)] = float(v)

# Create BQM and add the constant offset
bqm = dimod.BinaryQuadraticModel.from_qubo(Q_dict_clean)

# Add constant term to the BQM's offset
bqm.offset += const_offset

print(f"BQM created: {bqm.num_variables} variables, {len(bqm.quadratic)} interactions")
print(f"BQM offset (constant term): {bqm.offset:.2f}")

BQM created: 4004 variables, 254254 interactions
BQM offset (constant term): 385.00


### 3.1 Classical solvers

Simulated annealing

In [10]:
# Simplified A LOT because 100 reads takes too long
start_time = time.time()
sa_sampler = SimulatedAnnealingSampler()
sampleset_sa = sa_sampler.sample(bqm, num_reads=100, seed=42)
sa_runtime = time.time() - start_time

best_sample_sa = sampleset_sa.first.sample
best_energy_sa = sampleset_sa.first.energy

print(f"Runtime: {sa_runtime:.4f}s")
print(f"Best energy: {best_energy_sa:.6f}")
print(f"Number of reads: 100")
print(f"Top 5 solutions:")

# Convert generator into list
top5 = list(sampleset_sa.data(['sample', 'energy', 'num_occurrences'], sorted_by='energy'))[:5]
for i, (sample, energy, num_occurrences) in enumerate(top5):
    print(f"  {i+1}. Energy: {energy:.6f}, Occurrences: {num_occurrences}")

Runtime: 24.3998s
Best energy: -0.015513
Number of reads: 100
Top 5 solutions:
  1. Energy: -0.015513, Occurrences: 1
  2. Energy: -0.015188, Occurrences: 1
  3. Energy: -0.009112, Occurrences: 1
  4. Energy: -0.008460, Occurrences: 1
  5. Energy: -0.008352, Occurrences: 1


In [11]:
def extract_communities_from_sample(sample, N, k_communities):
    """
    Extract community assignments from QUBO sample.
    For node i, find c where x_{i,c} = 1
    """
    partition = np.zeros(N, dtype=int)
    
    for i in range(N):
        # Find which community this node is assigned to
        for c in range(k_communities):
            var_idx = i * k_communities + c
            if var_idx in sample and sample[var_idx] == 1:
                partition[i] = c
                break
        else:
            # If no community assigned (constraint violation), assign to community 0
            partition[i] = 0
    
    return partition
count = 1

for sample, energy, num_occurrences in top5:
    partition_sa = extract_communities_from_sample(sample, N, k_communities)
    modularity_sa = compute_modularity(partition_sa, A)
    num_communities_found_sa = len(np.unique(partition_sa))
    
    print(f"\nPartition from sample with energy {energy:.6f}:")
    print(f"\nSample {count}:")
    print(f"  Number of communities found: {num_communities_found_sa}")
    print(f"  Modularity: {modularity_sa:.6f}")
    print(f"  Community sizes: {sorted(np.bincount(partition_sa), reverse=True)}")
    count += 1


Partition from sample with energy -0.015513:

Sample 1:
  Number of communities found: 39
  Modularity: 0.046224
  Community sizes: [np.int64(6), np.int64(5), np.int64(4), np.int64(4), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]

Partition from sample with energy -0.015188:

Sample 2:
  Number of communities found: 42
  Modularity: 0.045573
  Community sizes: [np.int64(5), np.int64(4), np.int64(4), np.int64(3), np.i

### 3.2 Quantum/hybrid solvers

In [12]:
def greedy_refinement(A, partition):
    """
    Refines the partition using a greedy local search.
    Moves each node to the community that gives the highest modularity increase.
    """
    refined_partition = partition.copy()
    N = A.shape[0]
    unique_communities = np.unique(partition)
    improved = True
    
    current_modularity = compute_modularity(refined_partition, A)
    
    print(f"Starting refinement... Initial Modularity: {current_modularity:.6f}")
    
    while improved:
        improved = False
        nodes = np.random.permutation(N)
        
        for i in nodes:
            best_move = refined_partition[i]
            initial_comm = refined_partition[i]
            for target_comm in unique_communities:
                if target_comm == initial_comm:
                    continue
                refined_partition[i] = target_comm
                new_modularity = compute_modularity(refined_partition, A)
                if new_modularity > current_modularity:
                    current_modularity = new_modularity
                    best_move = target_comm
                    improved = True
            refined_partition[i] = best_move
            
    print(f"Refinement complete. Optimized Modularity: {current_modularity:.6f}")
    return refined_partition

def advanced_refinement(A, partition):
    N = A.shape[0]
    m = np.sum(A) / 2
    degrees = np.sum(A, axis=1)
    refined_partition = partition.copy()
    
    def get_delta_q(node_i, target_comm, curr_partition):
        # Calculate only the change in modularity for moving node_i
        # Internal edges to the target community
        comm_nodes = np.where(curr_partition == target_comm)[0]
        ki_in = np.sum(A[node_i, comm_nodes])
        ki = degrees[node_i]
        sum_tot = np.sum(degrees[comm_nodes])
        
        # Simplified Delta Q formula
        return (ki_in / m) - (sum_tot * ki) / (2 * m**2)

    improved = True
    while improved:
        improved = False
        nodes = np.random.permutation(N)
        for i in nodes:
            old_comm = refined_partition[i]
            # Only check communities that node i is actually connected to (saves time!)
            neighbor_nodes = np.where(A[i, :] > 0)[0]
            neighbor_comms = np.unique(refined_partition[neighbor_nodes])
            
            best_delta = 0
            best_move = old_comm
            
            for target_comm in neighbor_comms:
                if target_comm == old_comm: continue
                
                delta_q = get_delta_q(i, target_comm, refined_partition)
                if delta_q > best_delta:
                    best_delta = delta_q
                    best_move = target_comm
                    improved = True
            
            refined_partition[i] = best_move
            
    return refined_partition

3.2a D-Wave LeapHybrid (quantum+classical, requires Leap account)

In [13]:
start_time = time.time()
sampler = LeapHybridSampler()
sampleset_hybrid = sampler.sample(bqm, time_limit=10)

# target_edgelist = sampler.edgelist
# find_embedding(bqm.quadratic, target_edgelist)

# sampleset_hybrid = hybrid_sampler.sample(bqm, time_limit=1)
hybrid_runtime = time.time() - start_time

best_sample_hybrid = sampleset_hybrid.first.sample
best_energy_hybrid = sampleset_hybrid.first.energy
# use_hybrid = True

In [14]:
# Refine with greedy optimization
partition_hybrid_raw = extract_communities_from_sample(best_sample_hybrid, N, k_communities)
print("Applying greedy refinement to hybrid solution...")
partition_hybrid_optimized = greedy_refinement(A, partition_hybrid_raw)
partition_hybrid = extract_communities_from_sample(best_sample_hybrid, N, k_communities)

num_communities_found_hybrid = len(np.unique(partition_hybrid))
conductance_hybrid = compute_conductance(partition_hybrid, A)
mod_raw = compute_modularity(partition_hybrid_raw, A)
mod_opt = compute_modularity(partition_hybrid_optimized, A)

print(f"Runtime: {hybrid_runtime:.4f}s")
print(f"Best energy: {best_energy_hybrid:.6f}")
print(f"Number of communities found: {num_communities_found_hybrid}")
print(f"Original hybrid modularity: {mod_raw:.6f}")
print(f"Optimized hybrid modularity: {mod_opt:.6f}")
print(f"Conductance: {conductance_hybrid:.6f}")
print(f"Community sizes: {sorted(np.bincount(partition_hybrid), reverse=True)}")

Applying greedy refinement to hybrid solution...
Starting refinement... Initial Modularity: 0.069444
Refinement complete. Optimized Modularity: 0.890408
Runtime: 6.6268s
Best energy: -0.027127
Number of communities found: 39
Original hybrid modularity: 0.069444
Optimized hybrid modularity: 0.890408
Conductance: 0.866667
Community sizes: [np.int64(5), np.int64(4), np.int64(4), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0